# Indoor Object Detection with YOLO

Train a recent YOLO object detector on the Indoor Object Detection Dataset, using an 80/10/10 train/validation/test split with all seven classes represented in every split. The notebook reports validation mAP overall and per class, then visualizes good and bad validation predictions.

# Setup

This cell works locally and in Colab. If the notebook is opened directly in Colab without the repository files, it clones the project automatically.

In [1]:
import os
import subprocess
import sys
from pathlib import Path
from typing import Any, Mapping, Sequence

REPO_URL = "https://github.com/Abermal/Indoor-Object-Detection-Dataset.git"
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "pyproject.toml").exists():
    checkout = Path("/content/Indoor-Object-Detection-Dataset")
    if not checkout.exists():
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(checkout)])
    os.chdir(checkout)
    PROJECT_ROOT = checkout

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])

print(f"Working directory: {PROJECT_ROOT}")

CalledProcessError: Command '['D:\\Work\\interviews\\docu_sketch\\.venv\\Scripts\\python.exe', '-m', 'pip', 'install', '-q', '-e', '.']' returned non-zero exit status 1.

In [2]:
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from numpy.typing import NDArray
import pandas as pd
from PIL import Image
from ultralytics import YOLO

from indoor_object_detection import (
    CLASSES,
    DatasetSplit,
    ImageRecord,
    ZENODO_DOWNLOAD_URL,
    ensure_dataset,
    make_splits,
    parse_dlib_annotations,
    summarize_splits,
    write_yolo_dataset,
)

SEED = 42
MODEL_WEIGHTS = "yolo26n.pt"
EPOCHS = 30
IMAGE_SIZE = 640
BATCH_SIZE = 16
YOLO_DATA_DIR = Path("data/indoor_yolo")

print(f"Classes: {CLASSES}")
print(f"Dataset source: {ZENODO_DOWNLOAD_URL}")

ModuleNotFoundError: No module named 'matplotlib'

# Data Download and Parsing

If `Indoor Object Detection Dataset/` is already present, the helper uses it. Otherwise it downloads and extracts the Zenodo archive.

In [ ]:
dataset_root = ensure_dataset(PROJECT_ROOT)
records = parse_dlib_annotations(dataset_root)

class_counts = Counter(box.label for record in records for box in record.boxes)
dataset_summary = pd.DataFrame([
    {"images": len(records), "boxes": sum(len(record.boxes) for record in records), **class_counts}
]).T.rename(columns={0: "count"})

print(dataset_root)
dataset_summary

# Train/Validation/Test Split

The split is deterministic and preserves exact 80/10/10 image counts. To reduce leakage from consecutive video frames, each 10-frame temporal block stays in one split. The helper retries block assignments until every class appears in train, validation, and test.

In [ ]:
splits = make_splits(records, seed=SEED)
split_summary = summarize_splits(splits)
split_summary

In [ ]:
for split_name, split_records in splits.items():
    labels = {box.label for record in split_records for box in record.boxes}
    missing = set(CLASSES) - labels
    assert not missing, f"{split_name} is missing: {missing}"

print("All classes are present in train, validation, and test splits.")

# Convert to YOLO Format

Ultralytics expects images and normalized label text files. The conversion writes a fresh YOLO dataset directory and `data.yaml`.

In [ ]:
yolo_dir = write_yolo_dataset(splits, YOLO_DATA_DIR, overwrite=True)
data_yaml = yolo_dir / "data.yaml"
print(data_yaml)
print(data_yaml.read_text())

# Ground Truth Sanity Check

In [ ]:
def show_ground_truth(records_subset: Sequence[ImageRecord], n: int = 4) -> None:
    """Plot ground-truth boxes for the first ``n`` image records."""
    sample = records_subset[:n]
    fig, axes = plt.subplots(1, len(sample), figsize=(5 * len(sample), 4))
    if len(sample) == 1:
        axes = [axes]
    for ax, record in zip(axes, sample):
        image = Image.open(record.image_path).convert("RGB")
        ax.imshow(image)
        for box in record.boxes:
            rect = plt.Rectangle((box.x1, box.y1), box.x2 - box.x1, box.y2 - box.y1,
                                 fill=False, color="lime", linewidth=2)
            ax.add_patch(rect)
            ax.text(box.x1, box.y1, box.label, color="black", fontsize=9,
                    bbox={"facecolor": "lime", "alpha": 0.75, "pad": 1})
        ax.axis("off")
    plt.tight_layout()

show_ground_truth(splits[DatasetSplit.TRAIN], n=4)

# Train YOLO

`yolo26n.pt` is the current small Ultralytics checkpoint and is suitable for a Colab run. For a stronger final result, increase `EPOCHS`, use a larger YOLO26 checkpoint, and keep the same split.

In [ ]:
model = YOLO(MODEL_WEIGHTS)

train_results = model.train(
    data=str(data_yaml),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    seed=SEED,
    patience=8,
    workers=2,
    project="runs/detect",
    name="indoor_yolo_train",
    exist_ok=True,
)

best_weights = Path(model.trainer.best)
model = YOLO(best_weights)
print(f"Loaded best checkpoint: {best_weights}")

# Validation Metrics

This reports whole-validation-set metrics and class-level mAP. Ultralytics computes COCO-style `mAP50-95` plus `mAP50` and `mAP75`.

In [ ]:
metrics = model.val(
    data=str(data_yaml),
    split=DatasetSplit.VAL,
    imgsz=IMAGE_SIZE,
    project="runs/detect",
    name="indoor_yolo_val",
    exist_ok=True,
    plots=True,
)

overall_metrics = pd.DataFrame([{
    "precision": metrics.box.mp,
    "recall": metrics.box.mr,
    "mAP50": metrics.box.map50,
    "mAP75": metrics.box.map75,
    "mAP50-95": metrics.box.map,
}])
overall_metrics

In [ ]:
rows = []
names = metrics.names if hasattr(metrics, "names") else {idx: name for idx, name in enumerate(CLASSES)}
for class_id, class_name in names.items():
    try:
        precision, recall, map50, map5095 = metrics.box.class_result(int(class_id))
    except Exception:
        precision = recall = map50 = np.nan
        map5095 = metrics.box.maps[int(class_id)] if int(class_id) < len(metrics.box.maps) else np.nan
    rows.append({
        "class_id": int(class_id),
        "class": class_name,
        "precision": precision,
        "recall": recall,
        "mAP50": map50,
        "mAP50-95": map5095,
    })

per_class_metrics = pd.DataFrame(rows).sort_values("class_id").set_index("class")
per_class_metrics

# Good and Bad Validation Examples

Examples are ranked by a simple image-level score derived from class-aware IoU matching at IoU 0.5. Green boxes are ground truth and red boxes are predictions.

In [ ]:
def yolo_label_to_xyxy(
    label_row: Sequence[float], image_width: int, image_height: int
) -> tuple[int, NDArray[np.float64]]:
    """Convert one normalized YOLO label to pixel corner coordinates."""
    class_id, x_center, y_center, width, height = label_row
    x1 = (x_center - width / 2) * image_width
    y1 = (y_center - height / 2) * image_height
    x2 = (x_center + width / 2) * image_width
    y2 = (y_center + height / 2) * image_height
    return int(class_id), np.array([x1, y1, x2, y2], dtype=float)

def box_iou(a: NDArray[np.float64], b: NDArray[np.float64]) -> float:
    """Calculate intersection over union for two corner-format boxes."""
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    area_a = max(0, a[2] - a[0]) * max(0, a[3] - a[1])
    area_b = max(0, b[2] - b[0]) * max(0, b[3] - b[1])
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0

def read_yolo_labels(
    label_path: Path, image_width: int, image_height: int
) -> list[tuple[int, NDArray[np.float64]]]:
    """Read a YOLO label file and return pixel-coordinate boxes."""
    if not label_path.exists() or not label_path.read_text().strip():
        return []
    labels = []
    for line in label_path.read_text().strip().splitlines():
        values = np.array([float(x) for x in line.split()], dtype=float)
        labels.append(yolo_label_to_xyxy(values, image_width, image_height))
    return labels

def score_prediction(
    result: Any, label_path: Path, iou_threshold: float = 0.5
) -> dict[str, Any]:
    """Score one prediction using precision, recall, and matched IoU."""
    image_height, image_width = result.orig_shape
    gt = read_yolo_labels(label_path, image_width, image_height)
    pred_boxes = result.boxes.xyxy.cpu().numpy() if result.boxes is not None else np.empty((0, 4))
    pred_classes = result.boxes.cls.cpu().numpy().astype(int) if result.boxes is not None else np.empty((0,), dtype=int)
    used_predictions = set()
    matched_ious = []

    for gt_class, gt_box in gt:
        candidates = [
            (idx, box_iou(gt_box, pred_box))
            for idx, pred_box in enumerate(pred_boxes)
            if idx not in used_predictions and pred_classes[idx] == gt_class
        ]
        if candidates:
            idx, iou = max(candidates, key=lambda item: item[1])
            if iou >= iou_threshold:
                used_predictions.add(idx)
                matched_ious.append(iou)

    tp = len(matched_ious)
    precision = tp / len(pred_boxes) if len(pred_boxes) else 0.0
    recall = tp / len(gt) if gt else 1.0
    mean_iou = float(np.mean(matched_ious)) if matched_ious else 0.0
    score = 0.4 * precision + 0.4 * recall + 0.2 * mean_iou
    return {"score": score, "precision": precision, "recall": recall, "mean_iou": mean_iou, "gt": gt}

val_image_dir = yolo_dir / "images" / DatasetSplit.VAL
val_label_dir = yolo_dir / "labels" / DatasetSplit.VAL
val_images = sorted(val_image_dir.glob("*.jpg"))

predictions = model.predict(val_images, imgsz=IMAGE_SIZE, conf=0.25, iou=0.5, verbose=False)
ranked = []
for image_path, result in zip(val_images, predictions):
    label_path = val_label_dir / image_path.with_suffix(".txt").name
    item = score_prediction(result, label_path)
    item.update({"image_path": image_path, "result": result, "label_path": label_path})
    ranked.append(item)

ranked = sorted(ranked, key=lambda item: item["score"], reverse=True)
pd.DataFrame([{k: v for k, v in item.items() if k not in {"result", "gt"}} for item in ranked]).head()

In [ ]:
def draw_ranked_examples(
    examples: Sequence[Mapping[str, Any]], title: str
) -> None:
    """Plot ranked examples with ground-truth and predicted boxes."""
    fig, axes = plt.subplots(1, len(examples), figsize=(6 * len(examples), 5))
    if len(examples) == 1:
        axes = [axes]
    for ax, item in zip(axes, examples):
        image = Image.open(item["image_path"]).convert("RGB")
        ax.imshow(image)
        width, height = image.size

        for class_id, box in item["gt"]:
            rect = plt.Rectangle((box[0], box[1]), box[2] - box[0], box[3] - box[1],
                                 fill=False, color="lime", linewidth=2)
            ax.add_patch(rect)
            ax.text(box[0], box[1], f"GT {CLASSES[class_id]}", color="black", fontsize=8,
                    bbox={"facecolor": "lime", "alpha": 0.75, "pad": 1})

        result = item["result"]
        pred_boxes = result.boxes.xyxy.cpu().numpy() if result.boxes is not None else []
        pred_classes = result.boxes.cls.cpu().numpy().astype(int) if result.boxes is not None else []
        pred_conf = result.boxes.conf.cpu().numpy() if result.boxes is not None else []
        for pred_box, class_id, conf in zip(pred_boxes, pred_classes, pred_conf):
            rect = plt.Rectangle((pred_box[0], pred_box[1]), pred_box[2] - pred_box[0], pred_box[3] - pred_box[1],
                                 fill=False, color="red", linewidth=2)
            ax.add_patch(rect)
            ax.text(pred_box[0], pred_box[3], f"P {CLASSES[class_id]} {conf:.2f}", color="white", fontsize=8,
                    bbox={"facecolor": "red", "alpha": 0.75, "pad": 1})

        ax.set_title(f"score={item['score']:.2f}, P={item['precision']:.2f}, R={item['recall']:.2f}")
        ax.axis("off")
    fig.suptitle(title)
    plt.tight_layout()

draw_ranked_examples(ranked[:3], "Good validation examples")
draw_ranked_examples(ranked[-3:], "Bad validation examples")

# Test Set Evaluation

The assignment asks for validation metrics; this optional cell evaluates the held-out test split after model selection.

In [ ]:
test_metrics = model.val(data=str(data_yaml), split=DatasetSplit.TEST, imgsz=IMAGE_SIZE, name="indoor_yolo_test", exist_ok=True)
pd.DataFrame([{
    "precision": test_metrics.box.mp,
    "recall": test_metrics.box.mr,
    "mAP50": test_metrics.box.map50,
    "mAP75": test_metrics.box.map75,
    "mAP50-95": test_metrics.box.map,
}])